# 💰💰💰 <span style="color: white; background-color: Green"><b> Extração de Relatório do Contas a Pagar </b></span></p>

🧩 <span style="color: Lime"><b> 1- Realiza login e navegação automática no NetSuite </b></span></p>
- Abre o navegador  
- Acessa o NetSuite  
- Efetua login via Selenium  
- Navega pelos menus do sistema  
- Entra no relatório Registro de C/P (Contas a Pagar)  
- Executa a exportação do relatório para CSV
- Essa exportação é o ponto de partida para todo o pipeline

📥 <span style="color: Lime"><b> 2- Identifica e carrega automaticamente o arquivo baixado </b></span></p>
- Busca arquivos pelo padrão 'RegistrodeC_P' no diretório de Downloads
- Valida se o arquivo existe  
- Garante que não haja múltiplos arquivos conflitantes  
- Move arquivos já processados para uma pasta de histórico
- Essa etapa organiza a estrutura de arquivos e evita duplicidades na carga

🧹<span style="color: Lime"><b> 3- Processa e trata a base de Contas a Pagar </b></span></p>
- Após carregar o CSV
    - Conversão de encoding
- Tenta UTF-8, e se falhar, usa latin-1
    -Fundamental porque NetSuite às vezes exporta com codificação diferente
- Limpeza de totais
    - Linhas de TOTAL são removidas
- Padronização de colunas. Seleciona somente as colunas finais necessárias
    - Conta  
    - Tipo  
    - Data  
    - Numero_documento  
    - Id_fornecedor  
    - Memorando  
    - Faturado  
    - Pago
- Conversão de valores monetários
    - Valores como "R$ 1.234,56" → 1234.56.
- Limpeza de datas
    - Transforma strings → datetime
    - Remove datas inválidas

🏢 <span style="color: Lime"><b> 4- Valida fornecedores e cruza com tabela mestre </b></span></p>
- A automação carrega a planilha de mapeamento de fornecedores  
- Realiza merge para associar cada lançamento ao seu fornecedor  
- Reporta fornecedores ainda não cadastrados na base mestre (sinalizando para revisão)
- Isso mantém alinhamento entre contas a pagar e cadastro financeiro

📊 <span style="color: Lime"><b> 5- Gera o Excel final formatado </b></span></p>
- Cria o arquivo NETSUITE - CP.xlsx com
    - Aba única
    - Tabela estruturada (TableStyleLight13)
    - Rastreabilidade do processo
- Essa base é utilizada internamente no RH

📁 <span style="color: Lime"><b> 6- Atualiza automaticamente o relatório Excel oficial de Contas a Pagar </b></span></p>
- Abre o arquivo CONTAS A PAGAR.xlsx
- E via PyAutoGUI:
    - Navega entre abas  
    - Posiciona o cursor  
    - Dispara o refresh (Alt+F5)  
    - Aguarda atualização
- Isso garante que dashboards vinculados ao Excel sejam atualizados automaticamente

📈 <span style="color: Lime"><b> 7- Atualiza o Power BI Desktop — Contas a Pagar </b></span></p>
- Abre o arquivo CONTAS A PAGAR.pbix  
- Aguarda o carregamento  
- Executa o comando de Atualizar  
- Publica o relatório  
- Fecha a aplicação
- Essa etapa final atualiza o dashboard corporativo, tornando os dados disponíveis para a gestão.

🧾 <span style="color: Lime"><b> 8- Registra cada etapa no LOG do processo </b></span></p>
- PROCESSOS.xlsx  
- Recebe linhas contendo:
    - ID do processo  
    - Timestamp  
    - Etapa executada
- Essa rastreabilidade é essencial para auditoria e governança

⏱️ <span style="color: Lime"><b> 9- Mostra o resumo e o tempo total de execução </b></span></p>
- Ao final, o script exibe
    - Confirmação de sucesso  
    - Tempo total de execução  
    - Validações e alertas

# Importando as Bibliotecas

In [1]:
import gc
import logging
import numpy as np
import os
import pandas as pd
import pyautogui
import pyperclip
import shutil
import sys
import time
import tkinter as tk
import win32com.client
import win32gui, win32con
from asyncio.log import logger
from contextlib import contextmanager
import datetime
from dotenv import load_dotenv
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager

tempo_0 = [id, datetime.datetime.today(), 0]

# Logging (apenas console)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Caminhos de Pastas

In [2]:
CONFIG = {
    'id_processo': 12,
    'processos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'),
    'origem': Path(r'C:\Users\rodrigo.bernandes\Downloads'),
    'movidos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'),
    'saida': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\NETSUITE - CP.xlsx'),
    'fornecedores': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\FORNECEDORES.xlsx'),
    'env_path': Path(r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.3 - Automações\.env'),
    'padrao': 'RegistrodeC_P',
}

COLUNAS_SAIDA = [
    'Conta',
    'Tipo',
    'Data',
    'Numero_documento',
    'Id_fornecedor',
    'Memorando',
    'Faturado',
    'Pago'
]

# Registra Etapa do Processo

In [3]:
def append_registro_processo(caminho, id_proc, etapa):
    try:
        wb = load_workbook(caminho)
        ws = wb['REGISTROS']
        ws.append([id_proc, datetime.today(), etapa])
        wb.save(caminho)
        wb.close()
    except Exception as e:
        print(f"❌ Erro ao registrar etapa {etapa}: {e}")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Registro de processos

----------------------------------------------------------------------------------------------------


# Realizando o Login

In [4]:
# Configurações do Chrome para salvar o cache/perfil
chrome_options = Options()

# Caminho onde o perfil do robô será salvo
perfil_path = Path(r"C:\Users\rodrigo.bernandes\AppData\Local\Google\Chrome\User Data\AutomacaoNetSuite")

chrome_options.add_argument(f"user-data-dir={perfil_path}")
chrome_options.add_argument("--profile-directory=Default")

# Inicializa o navegador com as opções de perfil configuradas
navegador = webdriver.Chrome(options=chrome_options)

load_dotenv(dotenv_path='env_path')

netsuite_user = os.getenv("NETSUITE_USER")
netsuite_password = os.getenv("NETSUITE_PASSWORD")
netsuite_site = os.getenv("SITE_NETSUITE")

time.sleep(1)
window = win32gui.GetForegroundWindow()
win32gui.ShowWindow(window, win32con.SW_MAXIMIZE)
time.sleep(1)
navegador.get(netsuite_site)

try:
    elemento = navegador.find_element("name", "email")
    elemento.send_keys(netsuite_user)
    time.sleep(1)
    elemento = navegador.find_element("name", "password")
    time.sleep(2)
    elemento.send_keys(netsuite_password)
    elemento.send_keys(Keys.ENTER)

    print("⏳ Sincronizando NetSuite...")

except Exception as e:
    print(f"❌ Erro Geral: {e}")

finally:
    print(f"🏁 Acesso finalizado")

⏳ Sincronizando NetSuite...
🏁 Acesso finalizado


# Acessando o Relatório do Contas a Pagar

In [5]:
time.sleep(5)

# Acessando a aba
elemento = WebDriverWait(navegador, 10).until(
    EC.element_to_be_clickable((By.XPATH, "//a[@aria-label='Relatórios']"))
)
elemento.click()

# Clicar em Registro de C/P
elemento = WebDriverWait(navegador, 10).until(
    EC.element_to_be_clickable((By.XPATH, "//a[text()='Registro de C/P']"))
)
elemento.click()

time.sleep(10)

# Dentro de Registro de C/P

# Data atual
data_atual = datetime.date.today()

# Formato brasileiro: DD/MM/AAAA
fmt_br = "%d/%m/%Y"
data_atual_br = data_atual.strftime(fmt_br)

time.sleep(2)

# Data Até
elemento = navegador.find_element("name", "crit_1_to")
elemento.send_keys(Keys.CONTROL, 'a')
elemento.send_keys(Keys.DELETE)
elemento.send_keys(data_atual_br)

# Clicar em Atualizar
botao_atualizar = WebDriverWait(navegador, 10).until(
    EC.element_to_be_clickable((By.ID, "refresh"))
)
botao_atualizar.click()

time.sleep(14)

# Download do arquivo
botao_arquivo = WebDriverWait(navegador, 10).until(
    EC.element_to_be_clickable((By.ID, "csvbutton"))
)
botao_arquivo.click()

time.sleep(7)
pyautogui.hotkey("Alt", "F4")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Relatório Contas a Pagar extraído')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Relatório Contas a Pagar extraído

----------------------------------------------------------------------------------------------------


# Configuração de Logging

In [6]:
# Remover qualquer arquivo de log existente
log_file = Path('netsuite_cp.log')
if log_file.exists():
    try:
        log_file.unlink()
    except Exception as e:
        print(f"Erro ao remover arquivo de log existente: {e}")

# Resetar logging (remove todos os handlers antigos)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Configurar logging apenas no console (sem arquivo)
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False

# Remover handlers antigos do logger
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Adicionar apenas StreamHandler (console)
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)

# Carga e Tratamento da Base

In [7]:
from datetime import datetime

# --- Funções Auxiliares ---
@contextmanager
def gerenciar_workbook(caminho: Path, sheet: str):
    """Context manager para operacoes seguras com workbook."""
    wb = load_workbook(caminho)
    ws = wb[sheet]
    try:
        yield ws
    finally:
        wb.save(caminho)
        wb.close()

def registrar_etapa(caminho: Path, id_proc: int, etapa: int):
    """Registra etapa do processo."""
    try:
        with gerenciar_workbook(caminho, 'REGISTROS') as ws:
            ws.append([id_proc, datetime.today(), etapa])
        logger.info(f"[OK] Etapa {etapa} registrada com sucesso.")
    except Exception as e:
        logger.error(f"[ERRO] Falha ao registrar etapa {etapa}: {e}")

def carregar_de_para_fornecedores(caminho: Path) -> pd.DataFrame:
    """Carrega e processa de-para de fornecedores."""
    try:
        df = pd.read_excel(caminho)
        
        if 'ID' not in df.columns or 'NOME CADASTRO' not in df.columns:
            logger.warning("[AVISO] Colunas 'ID' ou 'NOME CADASTRO' nao encontradas no arquivo de fornecedores.")
            return pd.DataFrame() # Retorna DataFrame vazio se colunas essenciais faltarem
        
        df = df[['ID', 'NOME CADASTRO']].copy()
        df.columns = ['ID', 'Nome_cadastro']
        
        return df
    except FileNotFoundError:
        logger.error(f"[ERRO] Arquivo de fornecedores nao encontrado: {caminho}")
        return pd.DataFrame()
    except Exception as e:
        logger.error(f"[ERRO] Erro ao carregar de-para de fornecedores: {e}")
        return pd.DataFrame()

def converter_moeda(valor_str: str) -> float:
    """Converte string de moeda para float."""
    if pd.isna(valor_str) or valor_str == '':
        return 0.0
    
    try:
        valor = str(valor_str).replace('R$', '').replace('.', '').replace(',', '.')
        return float(valor)
    except ValueError:
        return 0.0
    except Exception as e:
        logger.warning(f"[AVISO] Erro ao converter valor '{valor_str}' para moeda: {e}. Retornando 0.0.")
        return 0.0

def processar_arquivo(caminho: Path, de_para: pd.DataFrame) -> pd.DataFrame:
    """Processa um arquivo NetSuite CP."""
    try:
        # --- 1. Leitura do CSV com fallback de encoding ---
        try:
            df = pd.read_csv(caminho, skiprows=6, on_bad_lines='skip', encoding='utf-8')
            logger.info(f"   [OK] Arquivo lido com encoding 'utf-8'.")
        except UnicodeDecodeError:
            df = pd.read_csv(caminho, skiprows=6, on_bad_lines='skip', encoding='latin-1')
            logger.info(f"   [OK] Arquivo lido com encoding 'latin-1' (fallback).")
        
        if df.empty:
            logger.warning(f"   [AVISO] Arquivo '{caminho.name}' esta vazio apos leitura.")
            return None

        # --- 2. Limpeza e Renomeação de Colunas ---
        df.columns = df.columns.str.strip() # Remove espaços em branco de todos os nomes de coluna
        
        mapeamento_colunas = {}
        for col in df.columns:
            col_lower = col.lower()
            # Renomear "Número do documento" para "Numero_documento"
            if 'nmero' in col_lower or 'número' in col_lower or 'documento' in col_lower:
                mapeamento_colunas[col] = 'Numero_documento'
            # Outras colunas que podem ter problemas de espaço ou encoding
            elif col_lower == 'data':
                mapeamento_colunas[col] = 'Data'
            elif col_lower == 'faturado':
                mapeamento_colunas[col] = 'Faturado'
            elif col_lower == 'pago':
                mapeamento_colunas[col] = 'Pago'
            elif col_lower == 'fornecedor':
                mapeamento_colunas[col] = 'Fornecedor'
            elif col_lower == 'memorando':
                mapeamento_colunas[col] = 'Memorando'
            elif col_lower == 'conta':
                mapeamento_colunas[col] = 'Conta'
            elif col_lower == 'tipo':
                mapeamento_colunas[col] = 'Tipo'

        if mapeamento_colunas:
            df = df.rename(columns=mapeamento_colunas)
            logger.info(f"   [OK] Colunas renomeadas: {mapeamento_colunas}")
        
        # Remover colunas duplicadas que podem ter surgido
        df = df.loc[:, ~df.columns.duplicated(keep='first')]
        logger.info(f"   [OK] Colunas após limpeza de duplicatas: {df.columns.tolist()}")

        # --- 3. Filtragem da Coluna 'Conta' ---
        if 'Conta' in df.columns:
            linhas_antes = len(df)
            # Remove apenas linhas onde 'Conta' começa com 'Total' (case-insensitive, strip spaces)
            df = df[~df['Conta'].astype(str).str.strip().str.lower().str.startswith('total', na=False)].copy()
            logger.info(f"   [OK] Removidas {linhas_antes - len(df)} linhas de totais da coluna 'Conta'.")
        else:
            logger.warning(f"   [AVISO] Coluna 'Conta' nao encontrada para filtragem no arquivo '{caminho.name}'.")

        # --- 4. Conversão de Tipos de Dados ---
        if 'Data' in df.columns:
            df['Data'] = pd.to_datetime(df['Data'], format='%d/%m/%Y', errors='coerce')
        if 'Faturado' in df.columns:
            df['Faturado'] = df['Faturado'].apply(converter_moeda)
        if 'Pago' in df.columns:
            df['Pago'] = df['Pago'].apply(converter_moeda)

        # --- 5. Merge com Fornecedores ---
        if 'Fornecedor' in df.columns and not de_para.empty:
            df = df.merge(de_para, left_on='Fornecedor', right_on='Nome_cadastro', how='left', suffixes=('', '_de_para'))
            
            # Limpar colunas com sufixo '_de_para' que podem ter sido criadas
            for col in df.columns:
                if col.endswith('_de_para'):
                    df = df.drop(columns=[col])
            
            df = df.rename(columns={'ID': 'Id_fornecedor'})
            df = df.drop('Nome_cadastro', axis=1, errors='ignore')
            logger.info(f"   [OK] Merge com fornecedores realizado. Colunas: {df.columns.tolist()}")
        elif 'Fornecedor' not in df.columns:
            logger.warning(f"   [AVISO] Coluna 'Fornecedor' nao encontrada no arquivo '{caminho.name}' para merge.")
        elif de_para.empty:
            logger.warning(f"   [AVISO] DataFrame de fornecedores vazio, merge nao realizado.")

        # --- 6. Limpeza Final e Preenchimento de Nulos ---
        df = df.loc[:, ~df.columns.duplicated(keep='first')] # Última limpeza de duplicatas
        df = df.fillna(0) # Preenche nulos com 0 (para numéricos)
        df = df.replace({pd.NA: None, np.nan: None, '<NA>': None, 'nan': None}) # Limpeza de outros tipos de nulos

        # --- 7. Seleção e Ordenação das Colunas Finais ---
        final_df_columns = []
        for col in COLUNAS_SAIDA:
            if col in df.columns:
                final_df_columns.append(col)
            else:
                # Adiciona colunas faltantes com valores padrão
                if col in ['Id_fornecedor', 'Faturado', 'Pago']:
                    df[col] = 0
                else:
                    df[col] = None
                final_df_columns.append(col)
                logger.warning(f"   [AVISO] Coluna '{col}' nao encontrada e adicionada com valores padrao.")
        
        df = df[final_df_columns] # Garante a seleção e ordem das colunas
        
        logger.info(f"   [OK] Processamento concluido para '{caminho.name}' com {len(df)} registros.")
        gc.collect() # Libera memória após processar cada arquivo
        return df
        
    except Exception as e:
        logger.error(f"   [ERRO] ERRO geral ao processar '{caminho.name}': {e}", exc_info=True)
        return None

def validar_novos_fornecedores(df: pd.DataFrame, de_para: pd.DataFrame) -> dict:
    """Valida se ha novos fornecedores nao cadastrados."""
    logger.info("\n[ETAPA] Validando fornecedores...")
    
    if 'Id_fornecedor' not in df.columns:
        logger.warning("[AVISO] Coluna 'Id_fornecedor' nao encontrada para validacao de fornecedores.")
        return {
            'total_registros': len(df),
            'registros_com_fornecedor': 0,
            'registros_sem_fornecedor': len(df),
            'novos_fornecedores_unicos': 0,
            'passou_validacao': False
        }

    novos_fornecedores = df[df['Id_fornecedor'] == 0.0].copy()
    
    validacao = {
        'total_registros': len(df),
        'registros_com_fornecedor': len(df[df['Id_fornecedor'] != 0.0]),
        'registros_sem_fornecedor': len(novos_fornecedores),
        'novos_fornecedores_unicos': novos_fornecedores['Fornecedor'].nunique() if 'Fornecedor' in novos_fornecedores.columns else 0,
        'passou_validacao': len(novos_fornecedores) == 0
    }
    
    logger.info(f"   [OK] Total de registros: {validacao['total_registros']}")
    logger.info(f"   [OK] Registros com fornecedor identificado: {validacao['registros_com_fornecedor']}")
    logger.info(f"   [OK] Registros sem fornecedor: {validacao['registros_sem_fornecedor']}")
    
    if validacao['novos_fornecedores_unicos'] > 0:
        logger.warning(f"\n   [AVISO] ATENCAO: {validacao['novos_fornecedores_unicos']} novo(s) fornecedor(es) encontrado(s)!")
        logger.warning("   [AVISO] Fornecedores nao cadastrados:")
        
        fornecedores_nao_cadastrados = novos_fornecedores['Fornecedor'].unique()
        for forn in fornecedores_nao_cadastrados:
            count = len(novos_fornecedores[novos_fornecedores['Fornecedor'] == forn])
            logger.warning(f"      * {forn} ({count} registro(s))")
    else:
        logger.info("\n   [OK] Validacao concluida: NENHUM novo fornecedor para cadastrar.")
        logger.info("   [OK] Todos os fornecedores estao devidamente cadastrados.")
    
    return validacao

def criar_excel_com_tabela(df: pd.DataFrame, caminho: Path):
    """Cria Excel com tabela formatada."""
    logger.info(f"\n[ETAPA] Criando Excel final ({len(df)} registros)...")
    
    try:
        df.to_excel(caminho, sheet_name='NETSUITE - CP', index=False, engine='openpyxl')
        
        wb = load_workbook(caminho)
        ws = wb.active
        
        tabela = Table(displayName="NETSUITE_CP", ref=ws.dimensions)
        estilo = TableStyleInfo(
            name="TableStyleLight13",
            showFirstColumn=False,
            showLastColumn=False,
            showRowStripes=True,
            showColumnStripes=True
        )
        tabela.tableStyleInfo = estilo
        ws.add_table(tabela)
        
        wb.save(caminho)
        wb.close()
        logger.info(f"[OK] Excel criado com sucesso em: {caminho}")
    except Exception as e:
        logger.error(f"[ERRO] Falha ao criar arquivo Excel: {e}", exc_info=True)
    finally:
        gc.collect() # Libera memória após criar o Excel

# --- Main Execution Block ---
if __name__ == "__main__":
    tempo_inicio = datetime.now()
    
    logger.info("=" * 80)
    logger.info("[INICIO] PROCESSAMENTO NETSUITE - CP")
    logger.info("=" * 80)
    
    try:
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 0)
        
        logger.info("\n[ETAPA 1] Buscando arquivos...")
        arquivos = sorted([f for f in CONFIG['origem'].iterdir() 
                          if f.name.startswith(CONFIG['padrao']) and f.suffix.lower() == '.csv'])
        
        if not arquivos:
            logger.error("[ERRO] Nenhum arquivo CSV encontrado na pasta de origem!")
            sys.exit(1) # Sai do script com erro
        
        logger.info(f"[OK] Encontrados {len(arquivos)} arquivo(s) CSV.")
        
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 1)
        
        logger.info("\n[ETAPA 2] Carregando de-para de fornecedores...")
        de_para = carregar_de_para_fornecedores(CONFIG['fornecedores'])
        if de_para.empty:
            logger.warning("[AVISO] Nao foi possivel carregar a base de fornecedores ou ela esta vazia.")
        else:
            logger.info(f"[OK] Carregados {len(de_para)} fornecedores.")
        
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 2)
        
        logger.info("\n[ETAPA 3] Processando arquivos...")
        
        todas_bases = []
        
        for idx, arquivo in enumerate(arquivos, 1):
            logger.info(f"\n   [ETAPA] Processando [{idx}/{len(arquivos)}] '{arquivo.name}'...")
            
            df = processar_arquivo(arquivo, de_para)
            
            if df is not None and not df.empty:
                todas_bases.append(df)
                logger.info(f"   [OK] {len(df)} registros adicionados de '{arquivo.name}'.")
                
                try:
                    destino = CONFIG['movidos'] / arquivo.name
                    shutil.move(str(arquivo), str(destino))
                    logger.info(f"   [OK] Arquivo '{arquivo.name}' movido para '{CONFIG['movidos'].name}'.")
                except Exception as e:
                    logger.warning(f"   [AVISO] Erro ao mover arquivo '{arquivo.name}': {e}")
            else:
                logger.warning(f"   [AVISO] Nenhum dado valido processado para '{arquivo.name}'.")
            gc.collect() # Limpa memória após cada iteração do loop
        
        registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 3)
        
        logger.info("\n[ETAPA 4] Consolidando dados...")
        
        if todas_bases:
            base_final = pd.concat(todas_bases, ignore_index=True)
            base_final = base_final.drop_duplicates()
            logger.info(f"[OK] {len(base_final)} registros consolidados e duplicatas removidas.")
            
            criar_excel_com_tabela(base_final, CONFIG['saida'])
            
            validacao = validar_novos_fornecedores(base_final, de_para)
            
            tempo_duracao = datetime.now() - tempo_inicio
            
            logger.info("\n" + "=" * 80)
            logger.info("[SUCESSO] PROCESSO FINALIZADO COM SUCESSO!")
            logger.info("=" * 80)
            logger.info(f"\n[RESUMO EXECUTIVO]:")
            logger.info(f"   * Arquivos processados: {len(arquivos)}")
            logger.info(f"   * Total de registros na base final: {validacao['total_registros']}")
            logger.info(f"   * Registros com fornecedor identificado: {validacao['registros_com_fornecedor']}")
            logger.info(f"   * Registros sem fornecedor: {validacao['registros_sem_fornecedor']}")
            
            if validacao['passou_validacao']:
                logger.info(f"\n[OK] VALIDACAO CONCLUIDA: Nenhum novo fornecedor para cadastrar.")
                logger.info(f"   Todos os arquivos foram movidos e tratados com sucesso.")
            else:
                logger.warning(f"\n[AVISO] ATENCAO: {validacao['novos_fornecedores_unicos']} novo(s) fornecedor(es) requer(em) cadastro.")
            
            logger.info(f"\n   Tempo de execucao: {tempo_duracao}")
            logger.info("=" * 80)
            
        else:
            logger.error("[ERRO] Nenhum arquivo foi processado ou resultou em dados validos para consolidacao!")
            sys.exit(1) # Sai do script com erro
        
    except Exception as e:
        logger.error(f"\n[ERRO CRITICO] Ocorreu um erro inesperado: {e}", exc_info=True)
        sys.exit(1) # Sai do script com erro
    finally:
        gc.collect() # Limpa memória no final da execução

2026-06-26 09:55:02,277 - INFO - ================================================================================
2026-06-26 09:55:02,289 - INFO - [INICIO] PROCESSAMENTO NETSUITE - CP
2026-06-26 09:55:02,300 - INFO - ================================================================================
2026-06-26 09:55:02,877 - INFO - [OK] Etapa 0 registrada com sucesso.
2026-06-26 09:55:02,878 - INFO - 
[ETAPA 1] Buscando arquivos...
2026-06-26 09:55:02,881 - INFO - [OK] Encontrados 1 arquivo(s) CSV.
2026-06-26 09:55:03,446 - INFO - [OK] Etapa 1 registrada com sucesso.
2026-06-26 09:55:03,448 - INFO - 
[ETAPA 2] Carregando de-para de fornecedores...
2026-06-26 09:55:03,590 - INFO - [OK] Carregados 178 fornecedores.
2026-06-26 09:55:04,062 - INFO - [OK] Etapa 2 registrada com sucesso.
2026-06-26 09:55:04,064 - INFO - 
[ETAPA 3] Processando arquivos...
2026-06-26 09:55:04,064 - INFO - 
   [ETAPA] Processando [1/1] 'RegistrodeC_P491.csv'...
2026-06-26 09:55:04,083 - INFO -    [OK] Arquivo lido

# Atualizando o Arquivo Excel CONTAS A PAGAR

In [8]:
time.sleep(5)
# Caminho do arquivo
path_excel = r"X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\CONTAS A PAGAR.xlsx"
os.startfile(path_excel) # Abre o arquivo 
time.sleep(5)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)
pyautogui.write('Contas_a_Pagar') # Digita o endereço completo
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)
pyautogui.hotkey('alt', 'f5') #Atualizando o arquivo
time.sleep(7)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
pyautogui.write('Tab.Din') # Digita o endereço completo
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)
pyautogui.hotkey('alt', 'f5') #Atualizando o arquivo
time.sleep(3)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
pyautogui.write('HC!B5')# Digita o endereço completo
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)
pyautogui.hotkey('alt', 'f5') #Atualizando o arquivo
time.sleep(8)

# Configuração de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

pyautogui.hotkey('ctrl', 'b') # Salvar o arquivo
time.sleep(5)
pyautogui.hotkey('alt', 'f4') # Fechar o arquivo
time.sleep(3)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Planilha atualizada com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Planilha atualizada com sucesso

----------------------------------------------------------------------------------------------------


# Atualização do Power BI - CONTAS A PAGAR

In [9]:
pyautogui.PAUSE = 1

# Entrar no Power BI
path_pbi = r"X:\Gestão de Pessoas\Analytics\09 - Power BI\CONTAS A PAGAR.pbix"
os.startfile(path_pbi) # Abre o arquivo
time.sleep(35)

# Atualizar Power BI
pyautogui.click(x=753, y=103) # Clicar em Atualizar
time.sleep(60)
pyautogui.click(x=1293, y=98) # Publicar
time.sleep(5)
pyautogui.click(x=863, y=477) # Salvar
time.sleep(5)
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(1)
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(7)
pyautogui.click(x=871, y=577) # Substituir
time.sleep(10)
pyautogui.click(x=990, y=585) # Clicar em Entendi
time.sleep(3)
pyautogui.hotkey("Alt", "F4")
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Power BI Atualizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Power BI Atualizado

----------------------------------------------------------------------------------------------------


# Atualizando o Arquivo Excel Controle HC e Atestados

In [10]:
# Caminho do arquivo
path_excel = r"X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx"
os.startfile(path_excel) # Abre o arquivo
time.sleep(60)
pyautogui.press('esc')
time.sleep(15)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)

# Digita o endereço completo
pyautogui.write('DRE_NETSUITE!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(5)
pyautogui.hotkey('alt', 'f5')
time.sleep(2)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)

# Digita o endereço completo
pyautogui.write('CONTAS_PAGAR!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(5)
pyautogui.hotkey('alt', 'f5')
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Planilha atualizada com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Planilha atualizada com sucesso

----------------------------------------------------------------------------------------------------


# Resumo de Finalização do Processo

In [11]:
tempo_1 = [id, datetime.today(), 8]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:05:54.115737

----------------------------------------------------------------------------------------------------
